# K-Means Clustering — 50,000 Data Points

This notebook follows the workflow from your original example: load `X` and `Y`, fit K-Means, get centroids, accept a new point, predict its cluster, and plot all clusters, centroids, and the new point.

It also includes dataset exploration, model selection using the elbow method and silhouette score, evaluation, and optional saving of the fitted model's cluster labels.

**Files needed:** Keep `kmeans_50000_data.csv` in the same folder as this notebook.

## 1. Import libraries

In [ ]:
# If required, uncomment and run:
# %pip install pandas numpy matplotlib scikit-learn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import joblib

RANDOM_STATE = 42

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv("kmeans_50000_data.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing values"))
print("\nDuplicate rows:", df.duplicated(subset=["X", "Y"]).sum())
print("\nDescriptive statistics:")
display(df[["X", "Y"]].describe())

## 3. Visualize the original data

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(df["X"], df["Y"], s=8, alpha=0.5)
plt.title("Original Data — 50,000 Points")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. Model selection: Elbow method and silhouette score

K-Means needs the number of clusters (`n_clusters`) to be chosen. We compare values of K from 2 to 8 using:
- **Inertia** for the elbow plot (lower is better, but it always decreases as K increases).
- **Silhouette score** (higher is generally better; computed on a reproducible sample to keep the notebook practical with 50,000 rows).

These are guidance tools, not guarantees that one K is universally correct.

In [ ]:
X = df[["X", "Y"]].dropna().copy()

# Sample for quicker silhouette scoring on a large dataset
sample_size = min(5000, len(X))
silhouette_sample = X.sample(n=sample_size, random_state=RANDOM_STATE)

k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    model.fit(X)
    inertias.append(model.inertia_)

    sample_labels = model.predict(silhouette_sample)
    silhouette_scores.append(
        silhouette_score(silhouette_sample, sample_labels)
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True)

axes[1].plot(list(k_values), silhouette_scores, marker="o")
axes[1].set_title("Silhouette Score (sample)")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette score")
axes[1].grid(True)

plt.tight_layout()
plt.show()

selection_table = pd.DataFrame({
    "K": list(k_values),
    "Inertia": inertias,
    "Silhouette_Score": silhouette_scores
})
display(selection_table)

suggested_k = list(k_values)[int(np.argmax(silhouette_scores))]
print("K with highest sampled silhouette score:", suggested_k)
print("For this generated dataset, K=5 is a sensible choice because five groups were generated.")

## 5. Fit the final K-Means model

To match your original example, the final model uses **K = 5**. You can change this to the K you choose after reviewing the model-selection plots.

`random_state` makes results reproducible, and `n_init=10` runs several initializations and keeps the best solution.

In [ ]:
N_CLUSTERS = 5  # Change this if your model-selection results suggest another K

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10
)

df = df.dropna(subset=["X", "Y"]).copy()
df["Cluster"] = kmeans.fit_predict(df[["X", "Y"]])
centroids = kmeans.cluster_centers_

print("Centroids (X, Y):")
print(centroids)

print("\nNumber of points in each cluster:")
display(df["Cluster"].value_counts().sort_index().to_frame("Number of points"))

print("\nSample rows with cluster labels:")
display(df.head(10))

## 6. Evaluate the clustering

In [ ]:
# Inertia is the sum of squared distances from points to their assigned centroids.
print(f"Inertia: {kmeans.inertia_:.2f}")

# Silhouette score on a sample, to avoid expensive full-dataset scoring.
eval_sample = df[["X", "Y", "Cluster"]].sample(
    n=min(5000, len(df)), random_state=RANDOM_STATE
)
sample_silhouette = silhouette_score(
    eval_sample[["X", "Y"]],
    eval_sample["Cluster"]
)
print(f"Silhouette score on {len(eval_sample):,} sampled points: {sample_silhouette:.4f}")

## 7. Plot clusters and centroids

In [ ]:
plt.figure(figsize=(8, 7))
colors = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))

for i in range(N_CLUSTERS):
    cluster_points = df[df["Cluster"] == i]
    plt.scatter(
        cluster_points["X"], cluster_points["Y"],
        s=8, color=colors[i], alpha=0.55, label=f"Cluster {i}"
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    s=220, marker="X", c="black", edgecolor="white",
    linewidth=1.2, label="Centroids"
)

plt.title("K-Means Clustering — 50,000 Data Points")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 8. Accept a new point and predict its cluster

This matches the input-and-prediction portion of your original code. Enter any X and Y coordinates, then the model predicts the nearest learned cluster.

In [ ]:
x = float(input("Enter X coordinate of the new point: "))
y = float(input("Enter Y coordinate of the new point: "))

new_point = pd.DataFrame({"X": [x], "Y": [y]})
cluster_label = int(kmeans.predict(new_point[["X", "Y"]])[0])

print(f"The point ({x}, {y}) belongs to Cluster {cluster_label}")

## 9. Plot clusters, centroids, and the new point

In [ ]:
plt.figure(figsize=(8, 7))

for i in range(N_CLUSTERS):
    cluster_points = df[df["Cluster"] == i]
    plt.scatter(
        cluster_points["X"], cluster_points["Y"],
        s=8, color=colors[i], alpha=0.5, label=f"Cluster {i}"
    )

plt.scatter(
    centroids[:, 0], centroids[:, 1],
    s=220, marker="X", c="black", edgecolor="white",
    linewidth=1.2, label="Centroids"
)

plt.scatter(
    x, y, c="yellow", s=180, edgecolor="black",
    marker="*", label="New Point", zorder=5
)

plt.annotate(
    f"Cluster {cluster_label}",
    (x, y),
    xytext=(8, 8),
    textcoords="offset points",
    fontsize=11,
    fontweight="bold"
)

plt.title("K-Means Clustering with New Input Point")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 10. Save results and trained model

In [ ]:
# Save the dataset with predicted cluster IDs
output_csv = "kmeans_50000_with_clusters.csv"
df.to_csv(output_csv, index=False)
print(f"Saved clustered dataset to: {output_csv}")

# Save the fitted K-Means model for reuse
model_path = "kmeans_model.joblib"
joblib.dump(kmeans, model_path)
print(f"Saved trained model to: {model_path}")

## Notes

- K-Means is **unsupervised learning**: it discovers groups without using a target label.
- `Cluster 0`, `Cluster 1`, etc. are arbitrary identifiers; their numbering can change between runs or configurations.
- The `Original_Group` column is included only because this is synthetic practice data. The K-Means model deliberately uses only `X` and `Y`, just like your original example.
- The elbow method and silhouette score help choose K, but inspect the plots and the purpose of the data before deciding.
- The new point is assigned to the cluster whose learned centroid is nearest under K-Means' Euclidean-distance objective.